##Datenbereinigung - Projekt: Eniac_Rabattstrategie

Version 1

In [1]:
import pandas as pd

##1.Einlesen der Datenbanken (Original)

---






###1.1.orderlines.csv


In [2]:
url = "https://drive.google.com/file/d/11bPnYi-rm-7u0fY0bPQiQMANi42Heinw/view?usp=sharing" # orderlines.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_orderlines = pd.read_csv(path)

###1.2.orders.csv

In [3]:
url = "https://drive.google.com/file/d/1lnjRkOcjvKRnXTDy4EJs3RkLhbnT6riZ/view?usp=sharing" # orders.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_orders = pd.read_csv(path)

###1.3.brands.csv

In [ ]:
url = "https://drive.google.com/file/d/17nX88jyuIBHuv4yp8bVJIdcmLiZaO6vr/view?usp=sharing" # brands.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_brands = pd.read_csv(path)

###1.4.products.csv

In [ ]:
url = "https://drive.google.com/file/d/1pWbwYB39QWeMD7yEpe6QiDW5Cr3n_Cdb/view?usp=sharing" # products.csv
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
df_products = pd.read_csv(path)

##2.Copy erstellen

In [ ]:
#orderlines
copy_orderlines = df_orderlines.copy()

In [ ]:
#orders
copy_orders = df_orders.copy()

In [ ]:
#Brands
copy_brands = df_brands.copy()

In [ ]:
#products
copy_products = df_products.copy()

##products.csv

##3.Exploration der Daten

**Ergebnis:**
- desc hat 7 null Werte
- price hat 46 null Werte
- Type ist ein String (Kategorie/ Produkttyp-Code/ keine Rechenoperation nötig)
- Type hat 50 fehlende Werte => anschauen Produktkategorisierung in Tag 7
- in_stock ist der Lagerstatus

In [ ]:
copy_orderlines.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38


In [ ]:
copy_orders.head(5)

,order_id,created_date,total_paid,state
0,241319,2017-01-02 13:35:40,44.99,Cancelled
1,241423,2017-11-06 13:10:02,136.15,Completed
2,242832,2017-12-31 17:40:03,15.76,Completed
3,243330,2017-02-16 10:59:38,84.98,Completed
4,243784,2017-11-24 13:35:19,157.86,Cancelled


In [ ]:
copy_brands.head(5)

,short,long
0,8MO,8Mobility
1,ACM,Acme
2,ADN,Adonit
3,AII,Aiino
4,AKI,Akitio


In [ ]:
copy_products.head(5)

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364


**Ergebnis:**

- doppelte Zeilen können gelöscht werden, da keine Info verloren geht nur redundante Kopien.
- APP2302/ APP2302 hat einen seltsamen type => 1,02E+12. Ergebnis: Type enthält fehlerhafte/ korrupte Werte (Expodentialschreibweise). 1,02E+12 bedeutet 1,02 × 10^12 = 1.020.000.000.000 — eine sehr große Zahl in wissenschaftlicher Notation. Das ist typisch, wenn Excel eine sehr lange Zahl (z.B. eine ID wie 13855401 oder ähnlich) automatisch "schön" formatiert hat Fazit: Separat anschauen.
- price seltsam: APP2302: 26.155.941 APP2302: 237.559.421. Fazit. Datenfehler prüfen

In [ ]:
# price anschauen ggf. Fehler bei der Datenexport-Pipeline. Gibt es weitere Werte die so aussehen
copy_products['price'].str.count(r"\.").value_counts()

,count
price,
0.0,11528
1.0,7321
2.0,431


In [ ]:
mult_decimal_rows = (copy_products['price'].str.count(r"\.") > 1).sum()
percent_corrupted = (100 * mult_decimal_rows / copy_products.shape[0])
print(f"{percent_corrupted:.2f}% der Zeilen haben mehrere Dezimalstellen im price")

2.23% der Zeilen haben mehrere Dezimalstellen im price


**Ergebnis:**

- 0 Punkte (11.528 Zeilen): wahrscheinlich Ganzzahl-Preise ohne Dezimalstelle (z.B. "59")
- 1 Punkt (7.321 Zeilen): normale Preise (z.B. "59.99")
- 2 Punkte (431 Zeilen): korrupte Preise wie Beispiel 26.155.941

In [ ]:
# 10 Beispiele von Korrupten Werten anschauen, um ein Muster zu erkennen
mask_corrupted = copy_products['price'].str.count(r"\.") > 1
copy_products.loc[mask_corrupted, 'price']

,price
665,1.639.792
792,4.694.994
797,4.090.042
827,2.199.791
885,5.609.698
...,...
19312,6.999.003
19313,6.999.003
19314,6.999.003
19315,6.999.003


**Ergebnis:**

- wenn der letzte Punkt ein Dezimalzeichen ist 1.639.792 => dann also 1639,79 €. Plausibel: Könnte Apple-Zubehör-Preis sein oder 6999.003  könnte High-End Technik sein
- Fazit: der letzte Punkt ist richtig, der andere falsch

In [ ]:
# SPÄTER ENTSCHEIDEN
#entfernt alle Punkte, außer dem letzten (Lookahead (?=.*\.) prüft: "kommt danach noch ein Punkt?").

In [ ]:
#????

**Ergebnis:**

- Type später anschauen

###3.1.[.info], usw.

In [ ]:
copy_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


In [ ]:
copy_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


In [ ]:
copy_brands.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   short   187 non-null    object
 1   long    187 non-null    object
dtypes: object(2)
memory usage: 3.1+ KB


In [ ]:
copy_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


In [ ]:
copy_products

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


In [ ]:
copy_products['sku'].unique()

array(['RAI0007', 'APP0023', 'APP0025', ..., 'THU0061', 'THU0062',
       'THU0063'], dtype=object)

##4.Duplikate

###4.1. Duplikate anschauen

In [ ]:
# Duplikate anschauen
mask_duplicated = copy_products.duplicated()
copy_products.loc[mask_duplicated]

,sku,name,desc,price,promo_price,in_stock,type
101,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
102,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
103,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
104,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
105,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
...,...,...,...,...,...,...,...
16831,APP2302,"Apple MacBook Pro 13 ""Core i5 Touch Bar 33GHz ...",New MacBook Pro 13-inch Core i5 Touch Bar 33 G...,26.155.941,26.155.941,0,"1,02E+12"
16833,APP2303,"Apple MacBook Pro 13 ""Core i5 Touch Bar 33GHz ...",New MacBook Pro 13 inch Touch Bar 33 GHz Core ...,237.559.421,23.755.942,0,"1,02E+12"
18190,PAR0077,Parrot Bebop Drone 2 Power,Drone cuadricóptero quality camera integrated ...,699.9,6.733.892,0,11905404
18308,NKI0010,Nokia Wireless sphygmomanometer Plata,Sphygmomanometer for iPhone iPad and iPod App.,129.99,1.149.899,1,11905404


### 4.2.Duplikate prüfen

In [ ]:
copy_orderlines.duplicated().sum()

np.int64(0)

In [ ]:
copy_products.duplicated().sum()

np.int64(8746)

In [ ]:
num_duplicated = copy_products.duplicated().sum()
total_rows = copy_products.shape[0]
percent_duplicated = (100 * num_duplicated / total_rows)
print(f"{percent_duplicated:.2f}% der Zeilen sind Duplikate")

45.26% der Zeilen sind Duplikate


In [ ]:
copy_products.duplicated().value_counts(normalize=True)

,proportion
False,0.547449
True,0.452551


**Ergebnis:**
- über 45% der Zeilen fehlen.

###4.3.Duplikate entfernen

In [ ]:
copy_orders.duplicated().sum()

np.int64(0)

In [ ]:
copy_orderlines.duplicated().sum()

np.int64(0)

**Ergebnis**:

In keinem der beiden DataFrames gibt es doppelte Zeilen.

In [ ]:
# Duplikate löschen
copy_products = copy_products.drop_duplicates()

In [ ]:
# mit shape prüfen wieviele Zeilen nach .drop_duplicates() übrig sind
copy_products.shape

(10580, 7)

**Ergebnis:**

- Nach Duplikat-Entfernung hat products 10.580 Zeilen, davon 377 (3,56%) mit korrupten Mehrfach-Punkt-Preisen.

##5.Datentypen prüfen

###5.1.Bestellungen_created date

In [ ]:
copy_orders["created_date"] = pd.to_datetime(copy_orders["created_date"])

Ergebnis:

- Wurde zum Datentyp Datum/Uhrzeit. Hinweis später erneut prüfen.

###5.2.Auftragszeilen

####5.2.1.Date

In [ ]:
copy_orderlines["date"] = pd.to_datetime(copy_orderlines["date"])

**Ergebnis:**

- date :sollte vom Datentyp Datum/Uhrzeit sein. Vorab prüfen später umstellen
- unit_pricesollte vom Datentyp float sein.

#### 5.2.2.Einzelpreis (price) prüfen (float)

**Ergebnis:**

- viele falsche Daten

=> Prüfen:

Große Stichprobe erstellen, um zu prüfen, ob alle Zahlen in der promo_priceSpalte tatsächlich entweder zwei oder drei Nachkommastellen haben.

In [ ]:
# in price
# Werte prüfen welche von 2 oder 3 Dezimalstellen betroffen sind

price_problems_number = copy_products.loc[(copy_products.price.str.contains(r"\d+\.\d+\.\d+"))|(copy_products.price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
price_problems_number

542

In [ ]:
print(f"The column price has in total {price_problems_number} wrong values. This is {round(((price_problems_number / copy_products.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column price has in total 542 wrong values. This is 5.12% of the rows of the DataFrame


In [ ]:
copy_products.sample ( 10 )

,sku,name,desc,price,promo_price,in_stock,type
17367,JYB0007,X3 Jaybird Wireless Bluetooth Headset Black,In-ear headphones waterproof and wireless 8 ho...,129,1.069.894,0,5384
15148,LAC0123-A,Open - LaCie d2 Quadra 4TB External Hard Drive...,4TB External Hard Drive USB 3.0 eSATA and Fire...,249,2.119.951,0,1298
16967,LAC0182-A,Open - LaCie Porsche Design Hard Drive 4TB USB...,Refurbished desktop hard drive with 4TB USB-C ...,179.99,148.026,0,11935397
17129,APP2073-A,"Open - Apple MacBook Pro 13 ""Core i5 23GHz | 8...",MacBook reconditioned 13 inch i5 23GHz with 8G...,1505.59,1.321.066,0,1298
19259,TPL0030-A,Open - TP-Link TL-PA4010P Passthrough Powerlin...,Refurbished Kit internet amplifiers with trans...,54.329,381.891,0,1334
10952,XTO0005-A,Open - Xtorm Power Case 4000mAh Battery Case B...,Case 4000 mAh battery for iPhone 6 Plus.,79,558.678,0,1298
19078,APP2606,Apple Loop Strap Sport 42mm Nacre,Durable and flexible strap sports lockable cli...,59,569.995,0,2449
11274,QNA0157,QNAP TS-853A | 4GB RAM Mac and PC Server NAS,8-bay NAS Server for Mac and PC.,966.79,9.659.902,0,12175397
15361,REP0310,GSM Antenna Repair iPhone 6 Plus,Repair service including parts and labor for i...,599.906,599.906,0,"1,44E+11"
18609,OTT0196,Otterbox USB-A to USB-C Cable 1 meter Twisted ...,Cable reinforced OtterBox USB flash charge-C t...,24.99,199.904,1,1325


**Ergebnis:**

- Die Spalte price enthält 542 falsche Werte
- Das sind 5.12 % die fehlerhaft sind.

In [ ]:
products_cl = copy_products.drop(columns=["promo_price"])

In [ ]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10580 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sku       10580 non-null  object
 1   name      10580 non-null  object
 2   desc      10573 non-null  object
 3   price     10534 non-null  object
 4   in_stock  10580 non-null  int64 
 5   type      10530 non-null  object
dtypes: int64(1), object(5)
memory usage: 578.6+ KB


**Ergebnis:**

- Offensichtlich ist es nun nicht mehr nötig, promo_price in einen numerischen Datentyp zu konvertieren.

**Ergebnis:**

- 5,15 % ist ein angemessener Anteil unserer Daten.
- Die Spalte „Preis“ ist jedoch wichtig, um Rabatte zu verstehen, daher muss sie absolut zuverlässig sein, da wir darauf unsere Geschäftsentscheidungen stützen.

=>Aus diesem Grund werden wir diese Zeilen löschen.

In [ ]:
#in promo_price
# Werte prüfen welche von 2 oder 3 Dezimalstellen betroffen sind

promo_problems_number = copy_products.loc[(copy_products.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(copy_products.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
promo_problems_number

9734

In [ ]:
print(f"The column promo_price has in total {promo_problems_number} wrong values. This is {round(((promo_problems_number / copy_products.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9734 wrong values. This is 92.0% of the rows of the DataFrame


**Ergebnis:**

- Die Spalte promo_price hat 9734 falsche Werte.
- Das sind 92% der Daten.

In [ ]:

copy_products = copy_products.dropna(subset='price')

In [ ]:
copy_products.shape[0]

10534

In [ ]:
copy_products.head()

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364


In [ ]:
# Filtert copy_products auf alle Zeilen, bei denen desc fehlt (NaN).
copy_products.loc[copy_products['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN,107,814.659,0,1298
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN,199.99,1.441.174,0,11905404


In [ ]:
# Bei allen Zeilen mit fehlendem desc den Wert aus name einsetzen.
copy_products.loc[copy_products['desc'].isna(), 'desc'] = copy_products.loc[copy_products['desc'].isna(), 'name']

In [ ]:
#Die Anzahl der Zeilen, bei denen desc fehlt (NaN) zählen.
copy_products['desc'].isna().sum()

np.int64(0)

**Ergebnis:**

Zwei Möglichkeiten:

- Zeilen schnell und einfach entfernen.
- Alternativ
Die Beschreibung später zur Kategorisierung verwenden wollen, daher werden wir sie **vorerst nicht entfernen**.

In [ ]:
# Count the number of decimal points in the unit_price
copy_orderlines['unit_price'].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


In [ ]:
# in Prozent
copy_orderlines['unit_price'].str.count(r"\.").value_counts(normalize=True)

,proportion
unit_price,
1,0.876969
2,0.123031


In [ ]:
# Count the rows with more than one `.`
mult_decimal_rows = (copy_orderlines['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / copy_orderlines.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

12.30% of the rows in our DataFrame have multiple decimal points in the unit_price


**Ergebnis:**

- 36.000 Zeilen von diesem Problem betroffen
- 12,30 % der Zeilen in unserem DataFrame weisen mehrere Dezimalstellen im Einzelpreis auf => beträchtlicher Anteil des Datensatzes
- Wir müssen daher die Ordnungsnummern der Zeilen mit zwei Dezimalstellen ermitteln und anschließend alle zugehörigen Zeilen entfernen.

In [ ]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = copy_orderlines['unit_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = copy_orderlines.loc[multiple_decimal_mask, "id_order"]

# Keep only the rows that do not have multiple decimal points
copy_orderlines = copy_orderlines.loc[~copy_orderlines['id_order'].isin(corrupted_order_ids)]

In [ ]:
copy_orderlines.shape[0]

216250

**Ergebnis:**

- Wir haben noch 216.250 Zeilen in den Auftragspositionen zur Verfügung
- alle Preise mit zwei Dezimalstellen wurdenentfernt

In [ ]:
# unit_price in einen numerischen Datentyp umwandeln.
copy_orderlines["unit_price"] = pd.to_numeric(copy_orderlines["unit_price"])

In [ ]:
copy_orderlines.sample(5)

,id,id_order,product_id,product_quantity,sku,unit_price,date
93315,1297687,379800,0,1,AP20178,759.00,2017-07-20 00:11:53
72573,1260968,362000,0,1,APP1985,659.00,2017-06-03 18:01:38
277515,1624544,516273,0,1,OWC0040-2,183.99,2018-02-23 00:39:00
244941,1572272,495329,0,1,SAM0071,442.58,2018-01-21 03:31:23
21123,1167422,318193,0,2,PAC1164,450.99,2017-01-31 13:19:40


In [ ]:
copy_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216250 non-null  int64         
 1   id_order          216250 non-null  int64         
 2   product_id        216250 non-null  int64         
 3   product_quantity  216250 non-null  int64         
 4   sku               216250 non-null  object        
 5   unit_price        216250 non-null  float64       
 6   date              216250 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 13.2+ MB


**Ergebnis:**

- die Spalte unit_price wurde nun in den richtigen Datentyp umzuwandeln.

##6.Fehlende Werte prüfen

- .info werden fehlende Werte identifiziert => einzeln betrachten
- Entscheidung:
- Löschen — wenn es nur wenige sind und der Verlust unbedeutend ist
- Imputieren (auffüllen) — z.B. mit Durchschnittswert, oder einem Platzhalter wie "Unbekannt"
- Vorerst ignorieren — später noch eine bessere Lösung finden

Optional:
- Werte aus anderen DataFrames übernehmen (falls redundant)
- oder andere kreative Lösung

In [ ]:
# Ein paar Beispiele anschauen, bei denen 'price' fehlt

print("\nBeispiel-Zeilen mit fehlendem Preis:")
copy_products[copy_products['price'].isna()].head(5)


Beispiel-Zeilen mit fehlendem Preis:


,sku,name,desc,price,promo_price,in_stock,type


In [ ]:
# Analog Info: Die Berechnung der fehlenden Werte auf einen Blick
copy_products.isna().sum()

,0
sku,0
name,0
desc,0
price,0
promo_price,0
in_stock,0
type,50


**Ergebnis:**

- Weniger als 0.5% betroffen --> löschen

In [ ]:
percent_corrupted = (100 * (copy_products["price"].isna().sum()) / copy_products.shape[0])

print(f"{percent_corrupted:.2f}% of the rows in the products dataset have multiple decimal points in the price column.")

0.00% of the rows in the products dataset have multiple decimal points in the price column.


###6.1.Bestellungen (orders)

###5.2.Bestellpositionen (orderlines)

In [ ]:
#alle Zeilen mit mindestens einem fehlenden Wert (NaN) aus copy_orderlines entfernen.
copy_orderlines = copy_orderlines.dropna(axis=0)

In [ ]:
# Datentyp prüfen z.B. Objekt zum Datentyp Datum/Uhrzeit

- hat 5 fehlende Werte in total_paid
- created_date in Orders ist objekt und sollte zum Datentyp Datum/Uhrzeit werden

##7.Speichern 01_Data_cleaned

In [ ]:
#Bereinigte Daten speichern in Drive

from google.colab import files

copy_orders.to_csv("orders_cl.csv", index=False)
files.download("orders_cl.csv")

copy_orderlines.to_csv("orderlines_cl.csv", index=False)
files.download("orderlines_cl.csv")

copy_products.to_csv("products_cl.csv", index=False)
files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>